In [1]:
import sys
import pathlib

sys.path.append("../src")

from ome_tiff_processing.image.image import OmeTiffFile
from ome_tiff_processing.datasets.batch import BatchImageGenerator

leica_pyramidal_image = "../data/Leica1_36832_38432_1_3_1.ome.tiff"
tif = OmeTiffFile(leica_pyramidal_image)

In [2]:
tif

[Image 0]
|-> [Level 0] shape=(3, 4668, 1616), names=('S', 'Y', 'X')
|-> [Level 1] shape=(3, 1167, 404), names=('S', 'Y1', 'X1')
|-> [Level 2] shape=(3, 291, 101), names=('S', 'Y2', 'X2')

[Image 1]
|-> [Level 0] shape=(3, 38432, 36832), names=('S', 'Y', 'X')
|-> [Level 1] shape=(3, 9608, 9208), names=('S', 'Y1', 'X1')
|-> [Level 2] shape=(3, 2402, 2302), names=('S', 'Y2', 'X2')
|-> [Level 3] shape=(3, 600, 576), names=('S', 'Y3', 'X3')
|-> [Level 4] shape=(3, 150, 144), names=('S', 'Y4', 'X4')

# Batched Image Selection
Assuming you would like to sample the entire image, with no real concern for where (including whitespace)

In [3]:
big = BatchImageGenerator(tif, 1, 0)

big.generate_batch(
    batch_size=256,
    image_size={"X": 256, "Y": 256},
    output_shape=("X", "Y", "S"),
).shape

torch.Size([256, 256, 256, 3])

# Batched Selection with ROI
Given a dataset of ROIs, we'd like to generate batches from that too.

In [4]:
from ome_tiff_processing.image.region import RegionSet

In [ ]:
import pathlib
import xarray as xr
import torch


class ROIBatchImageGenerator:

    def __init__(
        self, 
        tif: OmeTiffFile,
        rois: pathlib.Path | str | xr.Dataset,
        image = 0,
        level = 0,
    ):

        self.tif = tif
        self.image = image
        self.level = level

        self.rois = self._validate_rois(rois)

    def _validate_rois(self, rois):
        if isinstance(rois, str) | isinstance(rois, pathlib.Path):
            rois = xr.load_dataset(rois)
            
        assert len(rois) > 1, ValueError("Needs to have ROIs, none found.")

        return RegionSet(rois)

    def _validate_image_size(self, image_size):

        if image_size is None:
            image_size = {"X": 256, "Y": 256}

        if not isinstance(image_size, dict) or not image_size:
            raise ValueError("image_size must be a non-empty dictionary.")

        named_shape = self.tif[self.image][self.level].named_shape

        for axis, size in image_size.items():

            if axis not in named_shape:
                raise ValueError(f"Unknown axis: {axis}")

            if not isinstance(size, int) or isinstance(size, bool) or size <= 0:
                raise ValueError(f"Invalid size for {axis}: {size}")

            if size > named_shape[axis]:
                raise ValueError(
                    f"{axis} size {size} exceeds image dimension "
                    f"{named_shape[axis]}"
                )

        return image_size, named_shape

    def generate_batch(
        self,
        batch_size=64,
        image_size=None,
        output_shape=("X", "Y", "S"),
    ):
        image_size, named_shape = roi_big._validate_image_size(image_size)

        # Sample coordinates once
        sampled = roi_big.rois.sample_regions(n=batch_size).astype(int)

        coords = {
            axis: sampled.sel(coordinate=axis).values
            for axis in image_size
        }

        # Extract patches
        result = np.stack([
            roi_big.tif.sel(
                image=roi_big.image,
                level=roi_big.level,
                output_shape=output_shape,
                **{
                    axis: (start, start + image_size[axis])
                    for axis, start in zip(image_size, starts)
                },
            ).to_numpy()
            for starts in zip(*coords.values())
        ])

        return torch.from_numpy(result)

In [20]:
roi_big = ROIBatchImageGenerator(
    tif, 
    "/home/main/Projects/2026/DEC/ome-tiff-processing/data/leica_image_1_level_4_roi.ncdf",
    1, 
    0,
)

In [22]:
import numpy as np

roi_big.generate_batch(
    batch_size=256,
    image_size=None,
    output_shape=("X", "Y", "S"),
)

256it [00:00, 613.06it/s]


tensor([[[[224, 222, 223],
          [224, 222, 223],
          [224, 222, 223],
          ...,
          [223, 221, 224],
          [224, 222, 225],
          [224, 222, 225]],

         [[224, 222, 223],
          [224, 222, 223],
          [225, 223, 224],
          ...,
          [223, 221, 224],
          [224, 222, 225],
          [225, 223, 226]],

         [[225, 223, 224],
          [226, 224, 225],
          [227, 225, 226],
          ...,
          [223, 221, 224],
          [225, 223, 226],
          [226, 224, 227]],

         ...,

         [[227, 225, 228],
          [227, 225, 228],
          [228, 226, 229],
          ...,
          [227, 222, 226],
          [226, 221, 225],
          [227, 222, 226]],

         [[225, 223, 226],
          [226, 224, 227],
          [226, 224, 227],
          ...,
          [226, 222, 226],
          [226, 221, 225],
          [227, 222, 226]],

         [[223, 221, 224],
          [224, 222, 225],
          [224, 222, 225],
         